# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [4]:
#drop null values
spaceship = spaceship.dropna()

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [5]:
#target variable (to predict)
y = spaceship['Transported']

#features (to use for prediction)
X = spaceship.drop(columns=['Transported'])

In [6]:
X.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
dtype: object

In [7]:
#Identify numerical and categorical features

num_features = X.select_dtypes(include=['int64', 'float64']).columns
categ_features = X.select_dtypes(include=['object', 'bool']).columns

In [8]:
#feature scaling for numerical features + encoding for categorical features

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features), #mean=0, std=1 models like KNN, SVM
        ('categ', OneHotEncoder(handle_unknown='ignore'), categ_features) #convert categorical to numerical (0/1)
    ])


In [9]:
#feature selecion: evaluate the importance of each feature

from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)

In [10]:
#pipeline: combine preprocessing and feature selection

from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessor),
        ('feature_selection', selector)
    ]
)


**Perform Train Test Split**

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, #80% train, 20% test
    random_state=42, #for reproducibility
    stratify=y #preserve the proportion of classes in train and test sets
)


In [12]:
#apply the pipeline to the training data

X_train_transformed = pipeline.fit_transform(X_train, y_train)

#apply the pipeline to the test data
X_test_transformed = pipeline.transform(X_test)

In [13]:
from sklearn.metrics import accuracy_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [14]:
#bagging classifier and pasting the transformed data
#each base estimator is a decision tree different and majority voting is used to combine their predictions

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100, #number of base estimators
    random_state=42
)

#train the model
bagging.fit(X_train_transformed, y_train)

#evaluate the model
acc_bag = evaluate_model(bagging, X_test_transformed, y_test)
print(f"Bagging Classifier Accuracy: {acc_bag:.4f}")


Bagging Classifier Accuracy: 0.7519


- Random Forests

In [15]:
#random forest classifier
#random subsample features for each tree so that the trees are less correlated and the ensemble generalizes better

from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=200, #number of trees
    random_state=42
)

#train the model
random_forest.fit(X_train_transformed, y_train)

#evaluate the model
acc_rf = evaluate_model(random_forest, X_test_transformed, y_test)
print(f"Random Forest Classifier Accuracy: {acc_rf:.4f}")

Random Forest Classifier Accuracy: 0.7587


- Gradient Boosting

In [16]:
#gradient boosting classifier
#model that builds trees sequentially, each trying to correct errors of the previous one

from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting = GradientBoostingClassifier(
    n_estimators=100, #number of boosting stages
    learning_rate=0.1,
    random_state=42
)

#train the model
gradient_boosting.fit(X_train_transformed, y_train)

#evaluate the model
acc_gb = evaluate_model(gradient_boosting, X_test_transformed, y_test)
print(f"Gradient Boosting Classifier Accuracy: {acc_gb:.4f}")

Gradient Boosting Classifier Accuracy: 0.7769


- Adaptive Boosting

In [17]:
#adaptive boosting classifier
#focuses on misclassified instances by assigning them higher weights and repeating the process

from sklearn.ensemble import AdaBoostClassifier

adaptive_boosting = AdaBoostClassifier(
    n_estimators=100, #number of boosting stages
    learning_rate=0.5, #how much learning happens at each stage
    random_state=42
)

#train the model
adaptive_boosting.fit(X_train_transformed, y_train)

#evaluate the model
acc_ada = evaluate_model(adaptive_boosting, X_test_transformed, y_test)
print(f"Adaptive Boosting Classifier Accuracy: {acc_ada:.4f}")

Adaptive Boosting Classifier Accuracy: 0.7564


Which model is the best and why?

In [ ]:
#Results

results = {
    "Bagging": acc_bag,
    "Random Forest": acc_rf,
    "Gradient Boosting": acc_gb,
    "AdaBoost": acc_ada
}

results

{'Bagging': 0.7518910741301059,
 'Random Forest': 0.7586989409984871,
 'Gradient Boosting': 0.7768532526475038,
 'AdaBoost': 0.75642965204236}

Gradient Boosting is the best performing model, achieving the highest accuracy (≈ 0.78).
This model improves predictions in a sequential manner by focusing on the errors made by previous models, which allows it to learn more complex patterns in the data.

While Random Forest and AdaBoost also show strong performance, Gradient Boosting provides the best balance between bias and variance for this dataset, resulting in superior predictive accuracy.